# 08 - Repeated small replay corruption (RC): 5% / 10% more Syn flips before every update
Pre-registration: `experiments/prereg_repeated_corruption.md` (design committed `6e2de78` before any code; the
"Clarifications before execution" were added with the code, before any run). Code: `s12_repeated.py` plus three
default-off s3 flags. **This is not the sweep**: the sweep flipped 75-100% of stored Syn labels once; here the
attacker flips **rho x 2,000 more** stored Syn rows (rho = 10%, then 5%) before **each** of 6 updates, and the flips
persist (30% / 60% after E6).

- **Schedule (CII, tag `rep6`).** E0 = BENIGN + Syn, then LDAP, MSSQL, NTP, SNMP, SSDP and TFTP, one class per
  update. The data is the capped 12-label rows: 9,312 attack rows per class and 7 disjoint benign portions.
- **Arms**, for each rho and each seed 42-44:
  - clean;
  - poisoned (gate only);
  - poisoned_repair (registered soft R1);
  - plus a forced soft-R1 **diagnostic** on the poisoned arm, which is never deployed.
- **Decisions use validation only.** The trajectory is each arm's deployed model's Syn validation recall after
  each update. Test (A0..A6 of every arm) is scored **once**, later, after the validation files are committed.
- **Stopping rules:**
  - The clean-arm pilot comes first. Proceed only if the clean arm accepts every update in >= 2 of 3 seeds, with
    one registered fallback.
  - Fixed horizon of 6 updates.
  - rho = 0.10 runs first. rho = 0.05 is skipped if the measured T4 time already exceeds 90 min.

**Inputs (Add Input).** Notebook 04's output, as for notebooks 05-07: `nids_out/` with `clean_*.parquet` and
`runs/cii/all12_cii_s42`. Settings: **Internet ON**, accelerator **GPU T4**.

**Expected time:** about 35-45 min of T4. Run cells 1-6 interactively, never with Save & Run All.

**If the session dies:** rerun cells 1-5. Every step resumes: a finished run is skipped, and an unfinished arm
restarts deterministically. Cells 7-8 do nothing unless `RESTORE` / `DECISIONS_COMMIT` are set.

In [ ]:
# 1. Locate code and the notebook-04 outputs (read only); everything is written under /kaggle/working/nids_out
import glob, json, os, subprocess, sys, time, zipfile
from pathlib import Path
import pandas as pd
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)

CLONE = Path("/kaggle/working/code")
if not CLONE.exists() and os.environ.get("NIDS_CLONE", "1") == "1":
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/DevChi1l/cicddos2019-continual-nids",
                    str(CLONE)], check=False)
CODE_DIR = os.environ.get("NIDS_CODE") or next(
    (str(Path(p).parent) for p in [str(CLONE / "s12_repeated.py")] +
     sorted(glob.glob("/kaggle/input/**/s12_repeated.py", recursive=True)) if Path(p).exists()), None)
assert CODE_DIR, "code with s12_repeated.py not found: turn Internet ON"
sys.path.insert(0, CODE_DIR)
import s5_explain as X
git = subprocess.run(["git", "-C", CODE_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("code:", CODE_DIR, "| git commit:", git)

SRC = X.find_dir("runs/cii/all12_cii_s42/config.json", "NIDS_RC_SRC")
assert SRC and (SRC / "clean_train.parquet").exists(), \
    "attach notebook 04's output: nids_out/ with clean_*.parquet and runs/cii/all12_cii_s42"
WORK = Path(os.environ.get("NIDS_WORK", "/kaggle/working/nids_out"))
RC = WORK / "rc"
print("source (read only):", SRC, "\nwrites:", WORK)

def run(*args):
    cmd = [sys.executable, str(Path(CODE_DIR) / "s12_repeated.py"), *map(str, args)]
    print("$", " ".join(cmd[1:]), flush=True)
    subprocess.run(cmd, check=True, cwd=CODE_DIR)

COMMON = ["--work", WORK]

In [ ]:
# 2. GPU check; copy the Stage 1 files (hash-checked against all12) and build Stage 1b tag rep6 (about 1 min).
#    Refuses if the protected val rows differ from the registered feasibility check.
#    In a RESTORE session run this AFTER cell 7: it rebuilds the restored pilot's Stage 1b tags.
import torch
assert torch.cuda.is_available(), "Settings -> Accelerator -> GPU T4"
print(torch.cuda.get_device_name(0), torch.__version__)
run("prepare", "--src", SRC, *COMMON)

In [ ]:
# 3. Pilot (stopping rule 1): clean arm only, seeds 42-44, validation only (about 3-5 min).
#    Proceed if the clean arm accepts all 6 updates in >= 2 of 3 seeds; else the one registered fallback.
t0 = time.time()
run("pilot", *COMMON)
p = json.loads((RC / "pilot.json").read_text())
for x in p["attempts"]:
    print(x["tag"], "| seeds accepting every update:", x["seeds_accepting_all"], "| rejections per class:", x["rejections_per_class"])
print(f"PROCEED = {p['proceed']} ({p['reason']}) | {time.time() - t0:.0f} s")
if not p["proceed"]:
    print("STOP HERE and report pilot.json: the registration says stop (do not change the design)")

In [ ]:
# 4. Full runs, validation only: rho 0.10 (seeds 42-44), then rho 0.05 unless the measured T4 time > 90 min
#    (about 25-35 min). Re-running this cell resumes; finished runs are skipped.
t0 = time.time()
run("run", *COMMON)
print(f"run: {time.time() - t0:.0f} s")

In [ ]:
# 5. Trajectories and the registered predictions RC1-RC4, from saved files (validation only)
run("summarize", *COMMON)
traj = pd.read_csv(RC / "trajectory_val.csv")
print("deployed model's Syn val recall after update t:")
display(traj.pivot_table(index=["rho", "seed", "arm"], columns="t", values="Syn_recall").round(3))
print("C1 decision per update (True = accepted):")
display(traj[traj.t > 0].pivot_table(index=["rho", "seed", "arm"], columns="t", values="c1_accepted", aggfunc="first"))

## STOP - download BOTH zips from the Output panel (cell 6 writes them)
- `rc_val.zip` holds the validation JSON/CSV only: no buffers, models or test. Send it; it is committed under
  `experiments/results/08_rc/`.
- `rc_restore.zip` holds the deployed models A0..A6, the validation files and a sha256 manifest, with no test.
  Keep it for a new session.

Test is scored once, and only after that commit: set `DECISIONS_COMMIT` in cell 8, then run cells 8-9.

If this session has closed, start a new one and run these cells, never cells 3-6:
1. cell 1;
2. cell 7, with `RESTORE = True` and the restore zip attached;
3. cell 2;
4. cells 8 and 9.

In [ ]:
# 6. Validation outputs: rc_val.zip (JSON/CSV only) and rc_restore.zip (deployed models + manifest); no test
assert not (RC / "test").exists(), "test was already scored in this folder"
run("zip-val", *COMMON, "--bundle", WORK.parent / "rc_val.zip")
run("bundle", *COMMON, "--bundle", WORK.parent / "rc_restore.zip")

In [ ]:
# 7. RESTORE (only in a NEW session after the old one closed; skip otherwise). Then run cell 2, then cells 8-9.
RESTORE = False
if RESTORE:
    man = [m for m in sorted(glob.glob("/kaggle/input/**/RESTORE_MANIFEST.json", recursive=True))
           if (Path(m).parent / "rc").is_dir()] or sorted(glob.glob("/kaggle/input/**/rc_restore.zip", recursive=True))
    assert man, "attach the dataset made from rc_restore.zip"
    b = Path(man[0])
    run("restore", *COMMON, "--bundle", b.parent if b.name == "RESTORE_MANIFEST.json" else b)
else:
    print("restore skipped (RESTORE = False)")

In [ ]:
# 8. Test, ONCE. Disabled until DECISIONS_COMMIT is the sha of the commit that holds rc_val.zip's files.
#    score-test refuses unless every validation file here is byte-identical to that commit's copy under
#    experiments/results/08_rc/ and every deployed model has the hash recorded in rc/summary.json.
DECISIONS_COMMIT = ""
if DECISIONS_COMMIT:
    subprocess.run(["git", "-C", CODE_DIR, "fetch", "--depth", "50", "origin", "main"], check=True)
    run("score-test", *COMMON, "--decisions-commit", DECISIONS_COMMIT, "--repo", CODE_DIR)
else:
    print("skipped: commit the validation files first, then set DECISIONS_COMMIT")

In [ ]:
# 9. Test files (after cell 8 ran; original or restore session)
tdir = RC / "test"
if not tdir.exists():
    print("no test scored yet")
else:
    dest = WORK.parent / "rc_test.zip"
    pick = sorted(p for p in tdir.rglob("*") if p.is_file())
    with zipfile.ZipFile(dest, "w", zipfile.ZIP_DEFLATED) as zf:
        for p in pick:
            zf.write(p, p.relative_to(WORK))
    print(f"{len(pick)} files -> {dest} ({dest.stat().st_size / 1e6:.2f} MB). Download it from the Output panel.")